In [ ]:
import gymnasium as gym
import gymnasium_env
import random

Note: you may need to restart the kernel to use updated packages.


/Users/mario/Documents/proj/cam/Blokus/venv/lib/python3.12/site-packages/gymnasium/envs/registration.py:642: UserWarning: WARN: Overriding environment gymnasium_env/Blokus-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [18]:
from copy import deepcopy

done = False
def encode_board(board):
    return ''.join(str(cell) for row in board for cell in row)

def decode_board(encoded_board, n):
    board = []
    for i in range(n):
        row = [int(encoded_board[i * n + j]) for j in range(n)]
        board.append(row)
    return board

def compute_optimal_score(board_size, num_players, max_depth): ## for each state, I want the optimal action
    best_action, optimal_score_v = {}, {}
    global different_states, function_calls, num_actions, max_actions, not_zero
    different_states, function_calls, num_actions, max_actions, not_zero = 0, 0, 0, 0, 0
    
    def optimal_score(state:gym.Env, depth, passed=False) -> int:
        global function_calls, different_states, num_actions, max_actions, not_zero
        function_calls += 1
        board_encoding = encode_board(state.board)
        if (board_encoding, state.current_player) in optimal_score_v:
            return optimal_score_v[board_encoding, state.current_player]
        if depth == 0:
            return 0
        different_states += 1
        possible_actions = state.possible_actions_efficient(state.current_player)
        num_actions += len(possible_actions)
        max_actions = max(max_actions, len(possible_actions))
        if len(possible_actions) == 0:
            if passed:
                return 0
            else:
                new_state = deepcopy(state)
                new_state.current_player = (new_state.current_player) % new_state.num_players + 1
                return -optimal_score(new_state, depth - 1, passed=True)
        not_zero += 1
        ret = float('-inf')
        for action in possible_actions:
            new_state = deepcopy(state)
            obs, reward, terminated, truncated, info = new_state.step(action)
            if truncated:
                assert False
            eval = optimal_score_v[encode_board(new_state.board), new_state.current_player] = optimal_score(new_state, depth - 1, terminated)
            if ret < -eval + reward:
                best_action[board_encoding, state.current_player] = action
                ret = -eval + reward
                if depth > 97:
                    print(ret, new_state._decode_action(action), 100 - depth, different_states)
                    print(f"Action average: {num_actions / different_states}")
                    print(f"Max actions: {max_actions}")
                    print(f"Average not zero{num_actions / not_zero}")
        return ret

    env = gym.make('gymnasium_env/Blokus-v0', board_size=board_size, num_players=num_players, render_mode='human', render_scale=10, disable_env_checker=True)
    env = env.unwrapped
    env.order_enforce = False
    env.reset()
    
    optimal_score_v[encode_board(env.board), 1] = optimal_score(env, max_depth, False)

    return best_action, optimal_score_v, different_states, function_calls

In [19]:
BOARD_SIZE = 5

env = gym.make('gymnasium_env/Blokus-v0', board_size=BOARD_SIZE, num_players=2, render_mode='human', render_scale=10, disable_env_checker=True)
env = env.unwrapped
env.order_enforce = False


best_action, optimal_score_v, different_states, function_calls = compute_optimal_score(board_size=BOARD_SIZE, num_players=2, max_depth=100)
print("different states", different_states)
print("function calls", function_calls)
print(len(best_action), len(optimal_score_v))
# assert different_states == len(best_action)
# assert different_states == len(optimal_score_v)
# print(minimax_value, different_states, function_calls)

11 (0, 0, 'W', 0) 2 57
Action average: 3.8596491228070176
Max actions: 82
Average not zero27.5
12 (0, 1, 'Z4', 0) 2 479
Action average: 1.0438413361169103
Max actions: 82
Average not zero7.6923076923076925
-7 (0, 4, 'I5', 1) 1 2705
Action average: 0.66728280961183
Max actions: 82
Average not zero4.277251184834123
1 (0, 0, 'W', 0) 2 2800
Action average: 0.7021428571428572
Max actions: 84
Average not zero4.283224400871459
8 (0, 1, 'Y', 0) 2 6844
Action average: 0.7082115721800117
Max actions: 84
Average not zero2.8196625945317044
-3 (1, 3, 'L5', 1) 1 27364
Action average: 0.7296447887735711
Max actions: 84
Average not zero2.700297538544766
8 (0, 0, 'W', 0) 2 27373
Action average: 0.7315602966426771
Max actions: 84
Average not zero2.707178585913208
11 (0, 1, 'Z4', 0) 2 27497
Action average: 0.7313888787867767
Max actions: 84
Average not zero2.711473641634084
12 (0, 1, 'W', 2) 2 27594
Action average: 0.7309922446908749
Max actions: 84
Average not zero2.712979152656355
3 (0, 0, 'W', 0) 2 30

In [26]:
env.reset()
action1 = best_action[encode_board(env.board), 1]
env.step(action1)
action2 = best_action[encode_board(env.board), 2]
env.step(action2)
action3 = best_action[encode_board(env.board), 1]
env.step(action3)
action4 = best_action[encode_board(env.board), 2]
env.step(action4)
print(action1, action2, action3, action4)

print(env._decode_action(action1))
print(env._decode_action(action2))
print(env._decode_action(action3))
print(env._decode_action(action4))
# print(best_action["0" * BOARD_SIZE * BOARD_SIZE, 1])
# print(env._decode_action(best_action["0" * BOARD_SIZE * BOARD_SIZE, 1]), optimal_score_v["0" * BOARD_SIZE * BOARD_SIZE, 1])
# # print(env._decode_action(best_action["1111100000000000", 2]), optimal_score_v["1111100000000000", 2])
# # action=best_action["1111100000000000", 2]

# print(action)


31 1418 693 841
(0, 0, 'L5', 0)
(3, 1, 'Y', 2)
(1, 3, 'T4', 1)
(2, 0, '2', 0)


In [27]:
action5 = best_action[encode_board(env.board), 1]
env.step(action5)
action6 = best_action[encode_board(env.board), 2]
env.step(action6)
print(action5, action6)
print(env._decode_action(action5))
print(env._decode_action(action6))

# import pickle

# with open('best_action.pkl', 'wb') as f:
#     pickle.dump(best_action, f, protocol=pickle.HIGHEST_PROTOCOL)


930 524
(2, 1, 'V3', 1)
(1, 1, 'T4', 0)


In [28]:
action7 = best_action[encode_board(env.board), 1]
env.step(action7)
action8 = best_action[encode_board(env.board), 2]
env.step(action8)
print(action7, action8)
print(env._decode_action(action7))
print(env._decode_action(action8))

# with open('optimal_score_v.pkl', 'wb') as f:
#     pickle.dump(optimal_score_v, f, protocol=pickle.HIGHEST_PROTOCOL)

1680 336
(4, 0, '1', 0)
(0, 4, '1', 0)
